In [1]:
import numpy as np
import cv2
import os
import sys
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.utils import save_image
import torch.nn as nn
from torch.utils.data import DataLoader

In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
torch.cuda.empty_cache()

# frames extraction

## synthetic foggy frames extracted

In [3]:
video_path = "/content/drive/MyDrive/Colab Notebooks/Highway_5_fog.mp4"

cap = cv2.VideoCapture(video_path)

train_end = int(0.7 * total_frames)
val_end = int(0.85 * total_frames)
count = 0
saved = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # sample every 5th frame
    if count % 5 != 0:
        count += 1
        continue
    filename = f"frame_{saved:05d}.jpg"
    if count < train_end:
        path = f"/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train/fog/{filename}"
    elif count < val_end:
        path = f"/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/val/fog/{filename}"
    else:
        path = f"/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/fog/{filename}"
    cv2.imwrite(path, frame)
    saved += 1
    count += 1
cap.release()

print(saved, "frames extracted and split")

1126 frames extracted and split


## synthetic clear frames extracted

In [8]:
video_path = "/content/drive/MyDrive/Colab Notebooks/Highway_5_RGB.mp4"

cap = cv2.VideoCapture(video_path)

train_end = int(0.7 * total_frames)
val_end = int(0.85 * total_frames)
count = 0
saved = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # sample every 5th frame
    if count % 5 != 0:
        count += 1
        continue
    filename = f"frame_{saved:05d}.jpg"
    if count < train_end:
        path = f"/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train/clear/{filename}"
    elif count < val_end:
        path = f"/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/val/clear/{filename}"
    else:
        path = f"/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/clear/{filename}"
    cv2.imwrite(path, frame)
    saved += 1
    count += 1
cap.release()

print(saved, "frames extracted and split")

1126 frames extracted and split


## real foggy frames extracted

In [4]:
video_path = "/content/drive/MyDrive/Colab Notebooks/imp_fog1.mp4"
out_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"

os.makedirs(out_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imwrite(f"{out_dir}/frame_{count:05d}.jpg", frame)
    count += 1
cap.release()

print(f"{count} frames extracted")

2536 frames extracted


# dataset loader

In [4]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

class DehazeDataset(Dataset):
  def __init__(self, fog_dir, clear_dir):
      self.fog_dir = fog_dir
      self.clear_dir = clear_dir
      self.images = sorted(os.listdir(fog_dir))

  def __len__(self):
      return len(self.images)

  def __getitem__(self, idx):
      name = self.images[idx]
      fog_path = os.path.join(self.fog_dir, name)
      clear_path = os.path.join(self.clear_dir, name)
      fog = cv2.imread(fog_path)
      clear = cv2.imread(clear_path)
      fog = cv2.cvtColor(fog, cv2.COLOR_BGR2RGB)
      clear = cv2.cvtColor(clear, cv2.COLOR_BGR2RGB)
      fog = transform(fog)
      clear = transform(clear)
      return fog, clear, name

# model definition

In [5]:
# clone the github Dehazeformer repo

%cd /content/drive/MyDrive/Colab\ Notebooks/ug_research
!git clone https://github.com/IDKiro/DehazeFormer.git

/content/drive/MyDrive/Colab Notebooks/ug_research
fatal: destination path 'DehazeFormer' already exists and is not an empty directory.


In [6]:
# loading pretrained weights

repo_path = '/content/drive/MyDrive/Colab Notebooks/ug_research/DehazeFormer'
sys.path.insert(0, repo_path)

from DehazeFormer.models.dehazeformer import dehazeformer_s

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = dehazeformer_s().to(device)

checkpoint = torch.load(
'/content/drive/MyDrive/Colab Notebooks/ug_research/DehazeFormer/pretrained/dehazeformer-s.pth',
map_location=device
)

model.load_state_dict(checkpoint['state_dict'], strict=False)

print("Pretrained weights loaded")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Pretrained weights loaded


# training

In [7]:
criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)

print("Loss and optimizer ready")

Loss and optimizer ready


In [9]:
from torch.utils.data import DataLoader

train_dataset = DehazeDataset(
    "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train/fog",
    "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train/clear"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

epochs = 5
scaler = torch.amp.GradScaler('cuda')

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for fog, clear, _ in train_loader:
    fog = fog.to(device, non_blocking=True)
    clear = clear.to(device, non_blocking=True)

    optimizer.zero_grad()

    with torch.amp.autocast('cuda'):
      output = model(fog)
      loss = criterion(output, clear)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    total_loss += loss.item()

  avg_loss = total_loss / len(train_loader)

  print(f"Epoch {epoch+1}/{epochs}  Loss: {avg_loss:.4f}")

Epoch 1/5  Loss: 0.1000
Epoch 2/5  Loss: 0.0754
Epoch 3/5  Loss: 0.0697
Epoch 4/5  Loss: 0.0672
Epoch 5/5  Loss: 0.0658


In [10]:
save_path = "/content/drive/MyDrive/Colab Notebooks/ug_research/dehazeformer_finetuned.pth"

torch.save(model.state_dict(), save_path)

print("Model saved to:", save_path)

Model saved to: /content/drive/MyDrive/Colab Notebooks/ug_research/dehazeformer_finetuned.pth


# testing

In [14]:
test_dataset = DehazeDataset(
    "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/fog",
    "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/clear"
)

test_loader = DataLoader(test_dataset, batch_size=1)

output_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/outputs/synthetic"
os.makedirs(output_dir, exist_ok=True)

model.eval()

with torch.no_grad():

    for fog, clear, name in test_loader:

        fog = fog.to(device)

        output = model(fog)

        save_image(output, f"{output_dir}/{name[0]}")

print("Synthetic test images saved.")

Synthetic test images saved.


In [11]:
import os
import cv2
import torch
from torchvision import transforms
import numpy as np

input_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"
output_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/outputs/real"

transform = transforms.ToTensor()

model.eval()

for i in range(1, 11):

  input_dir = os.path.join(input_root, f"rf_ds{i}")
  output_dir = os.path.join(output_root, f"rf_ds{i}")

  os.makedirs(output_dir, exist_ok=True)

  if not os.path.exists(input_dir):
      print(f"Skipping missing folder: {input_dir}")
      continue

  img_list = os.listdir(input_dir)

  print(f"Processing rf_ds{i} with {len(img_list)} images")

  for img_name in img_list:
    img_path = os.path.join(input_dir, img_name)

    img = cv2.imread(img_path)
    if img is None:
      print(f"Skipping invalid image: {img_path}")
      continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]
    h = h - h % 8
    w = w - w % 8
    img = img[:h, :w]

    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
      output = model(tensor)

    output = output.clamp(0, 1)
    output = output.squeeze().permute(1, 2, 0).cpu().numpy()
    output = (output * 255).astype(np.uint8)

    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    cv2.imwrite(os.path.join(output_dir, img_name), output)

  print(f"Finished rf_ds{i}")

print("All datasets processed.")

Processing rf_ds1 with 1725 images
Finished rf_ds1
Processing rf_ds2 with 735 images
Finished rf_ds2
Processing rf_ds3 with 618 images
Finished rf_ds3
Processing rf_ds4 with 1811 images
Finished rf_ds4
Processing rf_ds5 with 611 images
Finished rf_ds5
Processing rf_ds6 with 1189 images
Finished rf_ds6
Processing rf_ds7 with 2126 images
Finished rf_ds7
Processing rf_ds8 with 1183 images
Finished rf_ds8
Processing rf_ds9 with 1085 images
Finished rf_ds9
Processing rf_ds10 with 2536 images
Finished rf_ds10
All datasets processed.


# evaluation

In [17]:
# synthetic dataset

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

gt_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/clear"
pred_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/outputs/synthetic"

psnr_list = []
ssim_list = []

for name in os.listdir(gt_dir):
  gt_path = os.path.join(gt_dir, name)
  pred_path = os.path.join(pred_dir, name)

  if not os.path.exists(pred_path):
      continue

  gt = cv2.imread(gt_path)
  pred = cv2.imread(pred_path)

  gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))

  gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
  pred = cv2.cvtColor(pred, cv2.COLOR_BGR2RGB)

  gt = gt.astype(np.float32) / 255.0
  pred = pred.astype(np.float32) / 255.0

  psnr = peak_signal_noise_ratio(gt, pred, data_range=1.0)
  ssim = structural_similarity(gt, pred, channel_axis=2, data_range=1.0)

  psnr_list.append(psnr)
  ssim_list.append(ssim)

print("Average PSNR:", np.mean(psnr_list))
print("Average SSIM:", np.mean(ssim_list))

Average PSNR: 19.319664913034334
Average SSIM: 0.6881884


In [18]:
# real dataset

input_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"
output_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/outputs/real"

sharp_input = []
sharp_output = []

for name in os.listdir(input_dir):
  in_path = os.path.join(input_dir, name)
  out_path = os.path.join(output_dir, name)

  if not os.path.exists(out_path):
      continue

  inp = cv2.imread(in_path, 0)
  out = cv2.imread(out_path, 0)

  s1 = cv2.Laplacian(inp, cv2.CV_64F).var()
  s2 = cv2.Laplacian(out, cv2.CV_64F).var()

  sharp_input.append(s1)
  sharp_output.append(s2)

print("Average sharpness input:", np.mean(sharp_input))
print("Average sharpness output:", np.mean(sharp_output))

Average sharpness input: 388.4285418439013
Average sharpness output: 582.138272328907


In [12]:
input_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"
output_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/outputs/real"

results = []

for i in range(1, 11):

    input_dir = os.path.join(input_root, f"rf_ds{i}")
    output_dir = os.path.join(output_root, f"rf_ds{i}")

    if not os.path.exists(input_dir):
        print(f"Skipping missing folder: {input_dir}")
        continue

    sharp_input = []
    sharp_output = []
    contrast_input = []
    contrast_output = []

    img_list = [f for f in os.listdir(input_dir) if f.endswith(".jpg")]

    for name in img_list:

        in_path = os.path.join(input_dir, name)
        out_path = os.path.join(output_dir, name)

        if not os.path.exists(out_path):
            continue

        inp = cv2.imread(in_path, 0)
        out = cv2.imread(out_path, 0)

        if inp is None or out is None:
            continue

        # Sharpness
        s1 = cv2.Laplacian(inp, cv2.CV_64F).var()
        s2 = cv2.Laplacian(out, cv2.CV_64F).var()

        sharp_input.append(s1)
        sharp_output.append(s2)

        # Contrast
        c1 = np.std(inp)
        c2 = np.std(out)

        contrast_input.append(c1)
        contrast_output.append(c2)

    if len(sharp_input) == 0:
        print(f"No valid images in rf_ds{i}")
        continue

    # Averages
    s_in = np.mean(sharp_input)
    s_out = np.mean(sharp_output)

    c_in = np.mean(contrast_input)
    c_out = np.mean(contrast_output)

    print(f"\nrf_ds{i}")
    print(f"Sharpness: {s_in:.2f} -> {s_out:.2f}")
    print(f"Contrast : {c_in:.2f} -> {c_out:.2f}")

    results.append([i, s_in, s_out, c_in, c_out])



rf_ds1
Sharpness: 99.65 -> 386.74
Contrast : 62.92 -> 45.84

rf_ds2
Sharpness: 168.15 -> 434.93
Contrast : 59.65 -> 42.66

rf_ds3
Sharpness: 84.43 -> 306.97
Contrast : 62.15 -> 36.02

rf_ds4
Sharpness: 25.67 -> 168.24
Contrast : 56.60 -> 33.29

rf_ds5
Sharpness: 242.72 -> 423.96
Contrast : 57.20 -> 35.63

rf_ds6
Sharpness: 278.10 -> 499.46
Contrast : 59.71 -> 37.56

rf_ds7
Sharpness: 54.05 -> 191.38
Contrast : 47.32 -> 24.35

rf_ds8
Sharpness: 58.27 -> 213.07
Contrast : 62.32 -> 36.58

rf_ds9
Sharpness: 32.16 -> 118.69
Contrast : 48.10 -> 25.66

rf_ds10
Sharpness: 175.72 -> 474.88
Contrast : 66.75 -> 50.23


In [13]:
import pandas as pd

df = pd.DataFrame(results, columns=[
    "Dataset", "Sharp_input", "Sharp_output",
    "Contrast_input", "Contrast_output"
])

df.to_csv("/content/drive/MyDrive/Colab Notebooks/real_ds_metrics.csv", index=False)